# Clustering

In [ ]:
#general packages
import pandas as pd
import numpy as np
import scanpy as sc
from glob import glob
import matplotlib.pyplot as plt
import squidpy as sq
%config InlineBackend.figure_format='retina'

import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")

In [ ]:
from skimage.measure import regionprops
import tifffile as tf
import json
from pathlib import Path

def convert_pos_file_to_csv(filename: str, px_size: float):
        """
        Parses stage positions from a JSON `.pos` file and converts them to a CSV format.

        :param img_path: Path to the directory containing the `.pos` file.
        :param output_file_name: Optional output file name for the stitched image.
        :return: A DataFrame with position information and the output file path.
        """

        with open(filename) as f:
            content = json.load(f)

        positions = []
        for pos in content["map"]['StagePositions']["array"]:
            label = pos["Label"]["scalar"]
            cor = pos['DevicePositions']["array"]

            index = 1 if cor[0]["Device"]["scalar"] == 'Adaptive Focus Control Offset' else 0
            coordinates = cor[index]['Position_um']["array"]

            if len(coordinates) < 2:
                print(f"Warning: Position {label} has incomplete coordinates. Skipping.")
                continue

            posinfo = {
                'label': label,
                'x': coordinates[0],
                'xpx': round(coordinates[0] / px_size),
                'y': coordinates[1],
                'ypx': round(coordinates[1] / px_size)
            }
            positions.append(posinfo)
        df = pd.DataFrame(positions)
        df["label"] = [f"Pos{pos}" for pos in range(0,len(df))]
        return df
    
def getCellLoc(src, pos_df):
    #get information about masks
    pos_info = Path(src).name.split("_")[1]
    mask = tf.imread(src)
    regions = regionprops(mask)
    finaldict = dict()
    for props in regions:
        y0, x0 = props.centroid
        # Now need to identify the proper offset
        subsetpositionfile = pos_df[pos_df['label'] == pos_info]
        x_final = x0 + subsetpositionfile['xpx'].values[0]
        y_final = y0 + subsetpositionfile['ypx'].values[0]
        fullkey = 'Cell' + str(props.label) + '.0_' + pos_info[:3] + f"_{pos_info[3:]}"
        finaldict[fullkey] = [x0, y0, x_final, y_final, pos_info]
    return finaldict

In [ ]:
#create df from positions file
pos_df = convert_pos_file_to_csv("/groups/CaiLab/personal/Lex/raw/250203_mb_161genes/pos.pos", 0.103)

#read in all masks
srcs = glob("/groups/CaiLab/personal/Lex/raw/250203_mb_161genes/pyfish_tools/output/edges_deleted/*_z0.tif")

#fill dictionary 
newdict = dict()
for src in srcs: 
    tempdict = getCellLoc(src, pos_df)
    newdict.update(tempdict)

In [ ]:
#transpose and rename columns
CellLocationsAndPositions = pd.DataFrame(newdict).T
CellLocationsAndPositions = CellLocationsAndPositions.rename(columns = {0: "x_rel", 1: "y_rel", 2: "x", 3: "y", 4: "Pos"})

In [ ]:
# Read in data
data = pd.read_csv("/groups/CaiLab/personal/Lex/raw/250203_mb_161genes/pyfish_tools/output/genebycell_5/final_1.51.51.5_seed33_heg_svm_p20.0_diff0_fdr9.0/genebycell_1.csv", index_col=0)
#remove rows that doesn't correspond to a gene
data = data[~data.index.str.contains("fake")]

In [ ]:
# Convert to AnnData object
adata = sc.AnnData(data.T)
#only get coords matching adata
locs = CellLocationsAndPositions[CellLocationsAndPositions.index.isin(adata.obs_names)]
locs = locs.loc[adata.obs_names]

# Reorder the DataFrame to match the order of adata.obs_names
locs = locs.loc[adata.obs_names]
# Add spatial coordinates
adata.obsm["spatial"] = locs[["y","x"]].values

# For example, requiring each cell to have at least 500 total counts
sc.pp.filter_cells(adata, min_counts=25)
# Filter genes that are expressed in fewer than 1 cells
sc.pp.filter_genes(adata, min_cells=25)
# Filter cells that have fewer than n genes expressed
sc.pp.filter_cells(adata, min_genes=5)
# CPM normalization
sc.pp.normalize_total(adata, target_sum=1e6)
# Log-transform the data
sc.pp.log1p(adata)  
# Use N number of top genes (all since they are all marker genes)
sc.pp.highly_variable_genes(adata, n_top_genes=adata.n_vars, subset=True)
# Z-score normalize and clip any value beyond 10 sigmas
sc.pp.scale(adata, max_value=10)
# Perform PCA to reduce dimensions and keep using arpack for consistent solutions
sc.tl.pca(adata, svd_solver='arpack', n_comps=adata.n_vars-1)

# Get the explained variance ratio
explained_variance_ratio = adata.uns['pca']['variance_ratio']
# Calculate the cumulative sum of explained variance ratio
cumulative_variance_ratio = np.cumsum(explained_variance_ratio)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, alpha=0.9, color='#A7C7E7', lw=2)
plt.xlabel('Principal Component', size = 14)
plt.ylabel('Explained Variance Ratio', size = 14)
plt.title('')
# Customize the plot (adding black box edges)
plt.gca().spines['top'].set_color('black')
plt.gca().spines['bottom'].set_color('black')
plt.gca().spines['left'].set_color('black')
plt.gca().spines['right'].set_color('black')
# You can also set the thickness of the edges (optional)
plt.gca().spines['top'].set_linewidth(1.5)
plt.gca().spines['bottom'].set_linewidth(1.5)
plt.gca().spines['left'].set_linewidth(1.5)
plt.gca().spines['right'].set_linewidth(1.5)
#plt.axvline(num_pcs_90, ls = "--", color = "k")
# Save the plot as an SVG file
#plt.savefig('Variance_Ratio_Plot.svg', format='svg')
plt.show()

In [ ]:
# Generate neighborhood graph. Use only top PCs that gives >90% of variance
sc.pp.neighbors(adata, n_neighbors=50, n_pcs=30)
# Perform UMAP on neighborhood graph
sc.tl.umap(adata, min_dist=0.4, spread=1, random_state=42)
# Perform community based clustering using leiden on neighborhood graph
sc.tl.leiden(adata, resolution=0.8) 

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

# Automatically adjust palette size based on clusters
num_clusters = adata.obs['leiden'].nunique()
custom_palette = sns.color_palette("tab20", num_clusters)

# Plot UMAP with updated palette
sc.pl.umap(
    adata, 
    color=["leiden"], 
    palette=custom_palette,
    title="",
    #edgecolor='black',  
    #linewidth=0.2,  # Add edge around the dots
    frameon=True,  
    show=False  
)

# ax = plt.gca()
# handles, labels = ax.get_legend_handles_labels()

# for handle in handles:
#     handle.set_edgecolor('black')
#     handle.set_linewidth(0.5)

plt.tight_layout()  # Prevents overlapping of plot and legend
plt.show()

# Cell typing

In [ ]:
#load annotations data
annotations = pd.read_csv("/groups/CaiLab/personal/Lex/raw/250113_mb_BSpeg_xtra_potentialTriton/mouse_brain_extra/Xenium_mBrain_v1.1_metadata.csv")

In [ ]:
# Perform differential expression across clusters
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon') 

#look at dot plot for top 3 genes per cluster
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="leiden", standard_scale="var", n_genes=3
)

# Extract the results into a DataFrame
marker_genes = sc.get.rank_genes_groups_df(adata, group=None)

# Get unique marker genes
unique_marker_genes = marker_genes['names'].unique()

# Filter your annotation DataFrame for these marker genes
annotated_markers = annotations[annotations['Genes'].isin(unique_marker_genes)]

In [ ]:
import pandas as pd
import numpy as np

####################################################
# 0) Prerequisites
#    - You have already run:
#       sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
#    - marker_genes = sc.get.rank_genes_groups_df(adata, group=None)
#    - annotations: a DataFrame with columns ["Genes", "Annotation"] 
#                  (cell-type assignments per gene)
####################################################

# Parameters
top_n = 10  # how many genes to take per cluster
col_for_sorting = 'pvals_adj'  # or 'pvals' or 'scores'

####################################################
# 1) Collect top N genes per cluster from marker_genes
####################################################
top_gene_set = set()
for cluster_id in marker_genes['group'].unique():
    # Subset to the cluster
    cluster_df = marker_genes[marker_genes['group'] == cluster_id].copy()
    # Sort by ascending p-value
    cluster_df.sort_values(col_for_sorting, ascending=True, inplace=True)
    # Take top N (e.g. 5)
    cluster_top_genes = cluster_df.head(top_n)['names'].unique().tolist()
    top_gene_set.update(cluster_top_genes)

print(f"Collected top {top_n} marker genes from each cluster.")
print(f"Total unique genes in that set: {len(top_gene_set)}")

####################################################
# 2) Filter your annotation DataFrame to these genes
####################################################
annotation_filtered = annotations[annotations['Genes'].isin(top_gene_set)]
print(f"Annotations before filtering: {len(annotations)}")
print(f"Annotations after filtering: {len(annotation_filtered)}")

####################################################
# 3) Build dict: { cell_type -> set of top marker genes }
####################################################
celltype_to_genes = {}

for row in annotation_filtered.itertuples(index=False):
    gene = row.Genes
    annotation = row.Annotation
    
    # Handle multi-annotation (e.g. "Astrocytes, Oligo")
    if isinstance(annotation, str) and ',' in annotation:
        ct_list = [ct.strip() for ct in annotation.split(',')]
    else:
        ct_list = [annotation]
    
    for ct in ct_list:
        if ct not in celltype_to_genes:
            celltype_to_genes[ct] = set()
        celltype_to_genes[ct].add(gene)

####################################################
# 4) Summation-based cell-type assignment per cluster
####################################################
clusters = adata.obs['leiden'].unique()
cluster_annotation = {}

# Convert var_names to a set for membership checks
adata_genes = set(adata.var_names)

for cluster in clusters:
    # Cells in this cluster
    cluster_idx = (adata.obs['leiden'] == cluster)
    
    # We'll store sum of expression for each cell type
    celltype_expr_sums = {}
    
    for ct, gene_set in celltype_to_genes.items():
        # Intersect with the genes that still exist in adata
        valid_gene_list = list(gene_set.intersection(adata_genes))
        
        if len(valid_gene_list) == 0:
            celltype_expr_sums[ct] = 0.0
            continue
        
        # Subset expression for these cells & genes
        expr_matrix = adata[cluster_idx, valid_gene_list].X
        
        # Sum (if log1p, it's a sum of logs; still okay for relative comparison)
        total_expr = expr_matrix.sum()
        
        celltype_expr_sums[ct] = total_expr
    
    # Pick whichever cell type is highest
    best_cell_type = max(celltype_expr_sums, key=celltype_expr_sums.get)
    cluster_annotation[cluster] = best_cell_type

####################################################
# 5) Print/inspect results
####################################################
print("\nCluster -> Assigned Cell Type (using top marker genes):")
for clust in clusters:
    print(f"{clust} -> {cluster_annotation[clust]}")

# Optionally, map cluster_annotation back to each cell in adata.obs
# to label each cell with a 'cell_type'
cluster_to_celltype = pd.Series(cluster_annotation, name='cell_type')
adata.obs['cell_type'] = adata.obs['leiden'].map(cluster_to_celltype)

print("\nSample of adata.obs[['leiden', 'cell_type']]:")
print(adata.obs[['leiden', 'cell_type']].head())


In [ ]:
# Automatically adjust palette size based on clusters
num_clusters = adata.obs['leiden'].nunique()
custom_palette = sns.color_palette("tab20", num_clusters)

fig, ax = plt.subplots(figsize=(5, 5))  # Adjust figure size as needed

# Plot UMAP on the specified axes
sc.pl.umap(
    adata,
    color='leiden',
    palette=custom_palette,  # or category_colors if supported
    title='',
    legend_fontsize=8,
    legend_fontoutline=0.5,
    edgecolor='black',
    linewidth=0.2,
    show=False,
    ax=ax  # Specify the axes to plot on
)

handles, labels = ax.get_legend_handles_labels()

for handle in handles:
    handle.set_edgecolor('black')
    handle.set_linewidth(0.5)

sns.despine()
plt.tight_layout()

for handle in handles:
    handle.set_edgecolor('black')
    handle.set_linewidth(0.5)

plt.show()


In [ ]:
# Generate a custom palette with a size matching the unique cell types
num_cell_types = adata.obs['cell_type'].nunique()
custom_palette = sns.color_palette("tab20", num_cell_types)

# Create a dictionary to explicitly map each category to a color
category_colors = dict(zip(adata.obs['cell_type'].cat.categories, custom_palette))

fig, ax = plt.subplots(figsize=(10, 8))  # Adjust figure size as needed

# Plot UMAP on the specified axes
sc.pl.umap(
    adata,
    color='cell_type',
    palette=custom_palette,  # or category_colors if supported
    title='',
    legend_fontsize=8,
    legend_fontoutline=0.5,
    edgecolor='black',
    linewidth=0.2,
    show=False,
    ax=ax  # Specify the axes to plot on
)

handles, labels = ax.get_legend_handles_labels()

for handle in handles:
    handle.set_edgecolor('black')
    handle.set_linewidth(0.5)

sns.despine()
plt.tight_layout()
#plt.tight_layout()  # Prevents overlapping of plot and legend
# Save as SVG
output_svg_file = "umap_cell_types.svg"
plt.savefig(output_svg_file, format="svg", dpi=300)  # Specify format and resolution
plt.show()

# Squidpy

In [ ]:
#look at cell types
np.unique(adata.obs["cell_type"])

In [ ]:
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt  # Ensure pyplot is imported
import squidpy as sq

# Define the cell types you want to plot
cell_types_of_interest = [
    'CA1-ProS', 'CA2', 'CA3', 
    'Dentate gyrus granule cells', 
    'L5 IT CTX', 'L6 CT CTX']

# Define your custom darker pastel palette as a dictionary
custom_darker_pastel_palette = {
    "CA1-ProS": "#79add2",           # Darker Pastel Blue
    "CA2": "#ffb26e",                # Darker Pastel Orange
    "CA3": "#80c680",                # Darker Pastel Green
    "Dentate gyrus granule cells": "#ba9a93",  # Darker Pastel Brown
    "L5 IT CTX": "#a67bb9",          # Dark Pastel Purple
    "L6 CT CTX": "#d88ca0",          # Dark Pastel Pink
}

# Create an ordered list of colors matching the order of cell_types_of_interest
ordered_colors = [custom_darker_pastel_palette[ct] for ct in cell_types_of_interest]

# Convert the list into a ListedColormap
custom_cmap = ListedColormap(ordered_colors)

# Subset your adata to include only the cell types of interest
adata_subset = adata[adata.obs["cell_type"].isin(cell_types_of_interest), :]

# Plot using Squidpy's spatial_scatter with the custom ListedColormap
sq.pl.spatial_scatter(
    adata_subset, 
    color="cell_type",
    shape=None, 
    figsize=(10, 10), 
    size=12,
    palette=custom_cmap  # Pass the ListedColormap here
)

# Save the plot as an SVG file
plt.savefig("spatial_scatter.svg", format="svg")

In [ ]:
#output specific positions used
pd.DataFrame(adata_subset.obs_names).to_csv("stitch_positions.csv", index=False)

In [ ]:
#plot sepcific gene distribution
gene = ["Cbln4", "Cntnap5b"] # Replace with your actual gene name

sq.pl.spatial_scatter(
    adata,
    color=gene,
    cmap="viridis",  # Use a continuous colormap such as 'viridis'
    shape=None,
    figsize=(10, 10),
    size=12,
)

# Subcluster

In [ ]:
#isolate indicies of interest
iso = adata_subset.obs_names

In [ ]:
# Read in data
data = pd.read_csv("/groups/CaiLab/personal/Lex/raw/250203_mb_161genes/pyfish_tools/output/genebycell_5/final_1.51.51.5_seed33_heg_svm_p20.0_diff0_fdr9.0/genebycell_1.csv", index_col=0)
#remove rows that doesn't correspond to a gene
data = data[~data.index.str.contains("fake")]

In [ ]:
# Convert to AnnData object
adata = sc.AnnData(data.T)
#only get coords matching adata
locs = CellLocationsAndPositions[CellLocationsAndPositions.index.isin(adata.obs_names)]
locs = locs.loc[adata.obs_names]

# Reorder the DataFrame to match the order of adata.obs_names
locs = locs.loc[adata.obs_names]
# Add spatial coordinates
adata.obsm["spatial"] = locs[["y","x"]].values

In [ ]:
#parse out cells of interest
adata = adata[adata.obs_names.isin(iso)]

In [ ]:
# For example, requiring each cell to have at least 500 total counts
sc.pp.filter_cells(adata, min_counts=20)
# Filter genes that are expressed in fewer than 1 cells
sc.pp.filter_genes(adata, min_cells=20)
# Filter cells that have fewer than n genes expressed
sc.pp.filter_cells(adata, min_genes=5)
# CPM normalization
sc.pp.normalize_total(adata, target_sum=1e6)
# Log-transform the data
sc.pp.log1p(adata)  
# Use N number of top genes (all since they are all marker genes)
sc.pp.highly_variable_genes(adata, n_top_genes=adata.n_vars, subset=True)
# Z-score normalize and clip any value beyond 10 sigmas
sc.pp.scale(adata, max_value=10)
# Perform PCA to reduce dimensions and keep using arpack for consistent solutions
sc.tl.pca(adata, svd_solver='arpack', n_comps=adata.n_vars-1)

# Get the explained variance ratio
explained_variance_ratio = adata.uns['pca']['variance_ratio']
# Calculate the cumulative sum of explained variance ratio
cumulative_variance_ratio = np.cumsum(explained_variance_ratio)
# Find the number of components that account for 90% of the variance
#num_pcs_90 = np.argmax(cumulative_variance_ratio >= 0.99) + 1  # Add 1 because indices start at 0
#print(f"Number of principal components that account for 90% of the variance: {num_pcs_90}")

In [ ]:
# Generate neighborhood graph. Use only top PCs that gives >90% of variance
sc.pp.neighbors(adata, n_neighbors=50, n_pcs=10)
# Perform UMAP on neighborhood graph
sc.tl.umap(adata, min_dist=0.4, spread=1, random_state=42)
# Perform community based clustering using leiden on neighborhood graph
sc.tl.leiden(adata, resolution=0.2) 

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

# Automatically adjust palette size based on clusters
num_clusters = adata.obs['leiden'].nunique()
custom_palette = sns.color_palette("tab20", num_clusters)

# Plot UMAP with updated palette
sc.pl.umap(
    adata, 
    color=["leiden"], 
    palette=custom_palette,
    title="",
    #edgecolor='black',  
    #linewidth=0.2,  # Add edge around the dots
    frameon=True,  
    show=False  
)

# ax = plt.gca()
# handles, labels = ax.get_legend_handles_labels()

# for handle in handles:
#     handle.set_edgecolor('black')
#     handle.set_linewidth(0.5)

plt.tight_layout()  # Prevents overlapping of plot and legend
plt.show()


In [ ]:
# Perform differential expression across clusters
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon') 

#look at dot plot for top 3 genes per cluster
sc.pl.rank_genes_groups_dotplot(
    adata, groupby="leiden", standard_scale="var", n_genes=3
)

# Extract the results into a DataFrame
marker_genes = sc.get.rank_genes_groups_df(adata, group=None)

# Get unique marker genes
unique_marker_genes = marker_genes['names'].unique()

# Filter your annotation DataFrame for these marker genes
annotated_markers = annotations[annotations['Genes'].isin(unique_marker_genes)]

In [ ]:
import pandas as pd
import numpy as np

####################################################
# 0) Prerequisites
#    - You have already run:
#       sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
#    - marker_genes = sc.get.rank_genes_groups_df(adata, group=None)
#    - annotations: a DataFrame with columns ["Genes", "Annotation"] 
#                  (cell-type assignments per gene)
####################################################

# Parameters
top_n = 10  # how many genes to take per cluster
col_for_sorting = 'pvals_adj'  # or 'pvals' or 'scores'

####################################################
# 1) Collect top N genes per cluster from marker_genes
####################################################
top_gene_set = set()
for cluster_id in marker_genes['group'].unique():
    # Subset to the cluster
    cluster_df = marker_genes[marker_genes['group'] == cluster_id].copy()
    # Sort by ascending p-value
    cluster_df.sort_values(col_for_sorting, ascending=True, inplace=True)
    # Take top N (e.g. 5)
    cluster_top_genes = cluster_df.head(top_n)['names'].unique().tolist()
    top_gene_set.update(cluster_top_genes)

print(f"Collected top {top_n} marker genes from each cluster.")
print(f"Total unique genes in that set: {len(top_gene_set)}")

####################################################
# 2) Filter your annotation DataFrame to these genes
####################################################
annotation_filtered = annotations[annotations['Genes'].isin(top_gene_set)]
print(f"Annotations before filtering: {len(annotations)}")
print(f"Annotations after filtering: {len(annotation_filtered)}")

####################################################
# 3) Build dict: { cell_type -> set of top marker genes }
####################################################
celltype_to_genes = {}

for row in annotation_filtered.itertuples(index=False):
    gene = row.Genes
    annotation = row.Annotation
    
    # Handle multi-annotation (e.g. "Astrocytes, Oligo")
    if isinstance(annotation, str) and ',' in annotation:
        ct_list = [ct.strip() for ct in annotation.split(',')]
    else:
        ct_list = [annotation]
    
    for ct in ct_list:
        if ct not in celltype_to_genes:
            celltype_to_genes[ct] = set()
        celltype_to_genes[ct].add(gene)

####################################################
# 4) Summation-based cell-type assignment per cluster
####################################################
clusters = adata.obs['leiden'].unique()
cluster_annotation = {}

# Convert var_names to a set for membership checks
adata_genes = set(adata.var_names)

for cluster in clusters:
    # Cells in this cluster
    cluster_idx = (adata.obs['leiden'] == cluster)
    
    # We'll store sum of expression for each cell type
    celltype_expr_sums = {}
    
    for ct, gene_set in celltype_to_genes.items():
        # Intersect with the genes that still exist in adata
        valid_gene_list = list(gene_set.intersection(adata_genes))
        
        if len(valid_gene_list) == 0:
            celltype_expr_sums[ct] = 0.0
            continue
        
        # Subset expression for these cells & genes
        expr_matrix = adata[cluster_idx, valid_gene_list].X
        
        # Sum (if log1p, it's a sum of logs; still okay for relative comparison)
        total_expr = expr_matrix.sum()
        
        celltype_expr_sums[ct] = total_expr
    
    # Pick whichever cell type is highest
    best_cell_type = max(celltype_expr_sums, key=celltype_expr_sums.get)
    cluster_annotation[cluster] = best_cell_type

####################################################
# 5) Print/inspect results
####################################################
print("\nCluster -> Assigned Cell Type (using top marker genes):")
for clust in clusters:
    print(f"{clust} -> {cluster_annotation[clust]}")

# Optionally, map cluster_annotation back to each cell in adata.obs
# to label each cell with a 'cell_type'
cluster_to_celltype = pd.Series(cluster_annotation, name='cell_type')
adata.obs['cell_type'] = adata.obs['leiden'].map(cluster_to_celltype)

print("\nSample of adata.obs[['leiden', 'cell_type']]:")
print(adata.obs[['leiden', 'cell_type']].head())

In [ ]:
# Automatically adjust palette size based on clusters
num_clusters = adata.obs['leiden'].nunique()
custom_palette = sns.color_palette("tab20", num_clusters)

fig, ax = plt.subplots(figsize=(5, 5))  # Adjust figure size as needed

# Plot UMAP on the specified axes
sc.pl.umap(
    adata,
    color='leiden',
    palette=custom_palette,  # or category_colors if supported
    title='',
    legend_fontsize=8,
    legend_fontoutline=0.5,
    edgecolor='black',
    linewidth=0.2,
    show=False,
    ax=ax  # Specify the axes to plot on
)

handles, labels = ax.get_legend_handles_labels()

for handle in handles:
    handle.set_edgecolor('black')
    handle.set_linewidth(0.5)

sns.despine()
plt.tight_layout()

for handle in handles:
    handle.set_edgecolor('black')
    handle.set_linewidth(0.5)

plt.show()


In [ ]:
# Generate a custom palette with a size matching the unique cell types
num_cell_types = adata.obs['cell_type'].nunique()
custom_palette = sns.color_palette("tab20", num_cell_types)

# Create a dictionary to explicitly map each category to a color
category_colors = dict(zip(adata.obs['cell_type'].cat.categories, custom_palette))

fig, ax = plt.subplots(figsize=(10, 8))  # Adjust figure size as needed

# Plot UMAP on the specified axes
sc.pl.umap(
    adata,
    color='cell_type',
    palette=custom_palette,  # or category_colors if supported
    title='',
    legend_fontsize=8,
    legend_fontoutline=0.5,
    edgecolor='black',
    linewidth=0.2,
    show=False,
    ax=ax  # Specify the axes to plot on
)

handles, labels = ax.get_legend_handles_labels()

for handle in handles:
    handle.set_edgecolor('black')
    handle.set_linewidth(0.5)

sns.despine()
plt.tight_layout()
#plt.tight_layout()  # Prevents overlapping of plot and legend
# Save as SVG
output_svg_file = "umap_cell_types.svg"
plt.savefig(output_svg_file, format="svg", dpi=300)  # Specify format and resolution
plt.show()

In [ ]:
#look at unique cell types
np.unique(adata_subset.obs['cell_type'])

In [ ]:
# # Suppose you want to show only two cell types: "NP SUB" and "Astrocyte"
# cell_types_of_interest = ["Astrocytes", 'CA1-ProS', 'CA2', 'CA3', 
#        'Dentate gyrus granule cells']

# # Create a subset of the AnnData object that includes only those cells
# adata_subset = adata_subset[adata_subset.obs["cell_type"].isin(cell_types_of_interest), :]


# Now plot using the subsetted AnnData
sq.pl.spatial_scatter(
    adata_subset, 
    color="cell_type", 
    shape=None, 
    figsize=(10, 10), 
    size=10
)

# Save the plot as an SVG file
plt.savefig("spatial_scatter2.svg", format="svg")

# Spatial Map: If you want to color masks by cell-type colors from UMAP and Spatial scatter

In [ ]:
import tifffile as tf

#path
mask_path = "/groups/CaiLab/personal/Lex/raw/250203_mb_161genes/pyfish_tools/output/edges_deleted/MMStack_Pos5_z0.tif"
#read mask
mask = tf.imread(mask_path)
# Extract Pos information from filename (assuming consistent naming)
pos = int(mask_path.split("Pos")[1].split("_")[0]) 

#make copy
mask_copy = mask.copy().astype(np.int16)
#grab leiden labels
labels = pd.DataFrame(adata_subset.obs["cell_type"])

In [ ]:
plt.imshow(mask)
plt.show()

In [ ]:
# Parse cell_id and Pos from the index
labels.reset_index(inplace=True)
labels[['cell_id', 'pos']] = labels['index'].str.extract(r'Cell(\d+)\.0_Pos_(\d+)', expand=True).astype(int)
labels = labels[['cell_id', 'pos', 'cell_type']]

# Filter labels for the current Pos
labels = labels[labels['pos'] == pos]

#make cell type dictionary
celltype_def = dict(zip(labels['cell_id'], labels['cell_type']))

In [ ]:
labels

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define a label for the background, if it exists, e.g., 0 for background
background_label = 0

# Extract unique cluster IDs from the mask, excluding the background label
unique_clusters = np.unique(mask_copy)
unique_clusters = unique_clusters[unique_clusters != background_label]  # Remove background label from clusters

# Initialize the mask_colored with a black background
mask_colored = np.zeros((*mask_copy.shape, 3), dtype=float)  # Create an RGB array filled with zeros (black background)

# Apply the color map to the mask
for cluster_id in unique_clusters:
    if cluster_id in celltype_def:
        celltype = celltype_def[int(cluster_id)]
        mask_colored[mask_copy == cluster_id] = category_colors[celltype]
    else:
        # Assign black color to clusters not in celltype_def
        mask_colored[mask_copy == cluster_id] = [0, 0, 0]

# Display the mask with the black background and colored clusters
plt.figure(figsize=(10, 10))
plt.imshow(mask_colored)
plt.axis('off')  # Hide the axes
#plt.savefig("projected_labels_on_masks.svg", bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
## Do for all
from pathlib import Path
output_dir  = Path("/groups/CaiLab/personal/Lex/raw/250203_mb_161genes/pyfish_tools/output/spatial_mapped_masks")
output_dir.mkdir(parents=True, exist_ok=True)


for pos in range(244):
    try:
        mask_path = f"/groups/CaiLab/personal/Lex/raw/250203_mb_161genes/pyfish_tools/output/edges_deleted/MMStack_Pos{pos}_z0.tif"
        #read mask
        mask = tf.imread(mask_path)
        # Extract Pos information from filename (assuming consistent naming)
        pos = int(mask_path.split("Pos")[1].split("_")[0]) 
        
        #make copy
        mask_copy = mask.copy().astype(np.int16)
        #grab leiden labels
        labels = pd.DataFrame(adata_subset.obs["cell_type"])
    
        # Parse cell_id and Pos from the index
        labels.reset_index(inplace=True)
        labels[['cell_id', 'pos']] = labels['index'].str.extract(r'Cell(\d+)\.0_Pos_(\d+)', expand=True).astype(int)
        labels = labels[['cell_id', 'pos', 'cell_type']]
        
        # Filter labels for the current Pos
        labels = labels[labels['pos'] == pos]
       
        #make cell type dictionary
        celltype_def = dict(zip(labels['cell_id'], labels['cell_type']))
    
        # Define a label for the background, if it exists, e.g., 0 for background
        background_label = 0
        
        # Extract unique cluster IDs from the mask, excluding the background label
        unique_clusters = np.unique(mask_copy)
        unique_clusters = unique_clusters[unique_clusters != background_label]  # Remove background label from clusters
        
        # Initialize the mask_colored with a black background
        mask_colored = np.zeros((*mask_copy.shape, 3), dtype=float)  # Create an RGB array filled with zeros (black background)
        
        # Apply the color map to the mask
        for cluster_id in unique_clusters:
            if cluster_id in celltype_def:
                celltype = celltype_def[int(cluster_id)]
                mask_colored[mask_copy == cluster_id] = category_colors[celltype]
            else:
                # Assign black color to clusters not in celltype_def
                mask_colored[mask_copy == cluster_id] = [0, 0, 0]
        
        # Display the mask with the black background and colored clusters
        tf.imwrite(str(output_dir / f"MMStack_Pos{pos}.ome.tif"), mask_colored)
    except:
        continue